# Proyecto DAM — Cuaderno de análisis del detector de intrusiones

Este cuaderno **ya está escrito**: tu trabajo no es programarlo, sino **entenderlo, ponerlo a prueba y criticarlo**.

**Cómo usarlo**

1. Antes de empezar, genera tus datos en la terminal (usa tu número de lista como semilla):
   `python generar_logs.py --seed <tu_numero> --salida auth.log --etiquetas etiquetas.csv`
2. Ejecuta las celdas **en orden** (Mayús + Intro).
3. Las celdas marcadas con ❓ son preguntas: escribe tu respuesta en la celda de texto que hay debajo (doble clic para editarla).
4. Las celdas marcadas con 🧪 son experimentos: cambia **solo** el valor indicado, ejecuta y anota el resultado en la tabla final.

## 0. Preparación

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from logtools import cargar_etiquetas, cargar_log, construir_features

RUTA_LOG = "auth.log"
RUTA_ETIQUETAS = "etiquetas.csv"

## 1. Del texto del log a una tabla

La función `cargar_log` (en `logtools.py`) lee cada línea del fichero, extrae sus datos con expresiones regulares y los guarda en un `DataFrame`.

In [ ]:
df = cargar_log(RUTA_LOG)
df.head(10)

In [ ]:
df["tipo"].value_counts()

❓ **P1.** ¿Qué representa cada fila de `df`? ¿Qué columnas tiene y qué significa cada una?

**Tu respuesta:**

❓ **P2.** Abre `logtools.py` y localiza `PATRON_LINEA`. Explica con tus palabras qué trozo de la línea captura cada parte con nombre (`mes`, `dia`, `hora`, `host`, `pid`, `msg`).

**Tu respuesta:**

❓ **P3.** El log de syslog no trae el año. ¿Cómo lo resuelve el código y qué problema podría dar con un log que cruce el 31 de diciembre?

**Tu respuesta:**

## 2. Características por IP

Un modelo de IA no entiende líneas de texto: necesita **una fila de números por cada cosa que queremos clasificar**. Aquí clasificamos direcciones IP, así que `construir_features` resume lo que ha hecho cada IP.

In [ ]:
X = construir_features(df)
X.head()

In [ ]:
X.describe().T

❓ **P4.** ¿Qué representa ahora cada fila? ¿Por qué se agrupa por IP y no por usuario o por evento?

**Tu respuesta:**

❓ **P5.** Antes de mirar los resultados del modelo, elige **tres características** que creas que mejor distinguen a un atacante de un usuario normal y justifica por qué.

**Tu respuesta:**

## 3. ¿Cuántos atacantes hay y en qué se diferencian?

In [ ]:
y = cargar_etiquetas(RUTA_ETIQUETAS, X.index)
print(y.value_counts().rename({0: "legítimas", 1: "atacantes"}))
print(f"Proporción de atacantes: {y.mean():.0%}")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 3.5))
for eje, col in zip(ejes, ["n_fallos", "n_usuarios", "max_fallos_5min"]):
    X.assign(etiqueta=y).boxplot(column=col, by="etiqueta", ax=eje)
    eje.set_title(col)
    eje.set_xlabel("0 = legítima, 1 = atacante")
plt.suptitle("")
plt.tight_layout()
plt.show()

❓ **P6.** ¿Coinciden las tres características que elegiste en P5 con lo que ves en los gráficos? ¿Alguna te ha sorprendido?

**Tu respuesta:**

❓ **P7.** Hay IPs legítimas con muchos fallos y atacantes con pocos. Localiza al menos un caso de cada tipo en `X` (pista: `X[(y == 0) & (X["n_fallos"] > 10)]`) y explica por qué engañan a una regla sencilla.

**Tu respuesta:**

## 4. Entrenar y evaluar el modelo

Se reserva un 25 % de las IPs para **examinar** el modelo con datos que no ha visto durante el entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

modelo = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

❓ **P8.** Lee la matriz de confusión: ¿cuántos usuarios legítimos habrían sido bloqueados por error (falsos positivos)? ¿Y cuántos atacantes se han colado (falsos negativos)? ¿Qué error es más grave en un servidor real y por qué?

**Tu respuesta:**

❓ **P9.** ¿Qué hacen `stratify=y` y `class_weight="balanced"`? ¿Qué pasaría sin ellos?

**Tu respuesta:**

❓ **P10.** ¿Por qué se evalúa con IPs que el modelo no ha visto al entrenar? ¿Qué podría ocurrir si evaluáramos con las mismas?

**Tu respuesta:**

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=X.columns).sort_values()
importancias.plot.barh(figsize=(6, 4), title="Importancia de cada característica")
plt.tight_layout()
plt.show()

❓ **P11.** ¿Qué dos características pesan más? ¿Tiene sentido desde el punto de vista de un administrador? ¿Hay alguna que te haga sospechar (pista: ¿qué pasa si una característica solo existe cuando el ataque ya ha tenido éxito)?

**Tu respuesta:**

## 5. Herramientas para experimentar

Estas funciones ya están hechas. Las usarás en los experimentos de la sección 6: **no hace falta modificarlas**.

In [ ]:
def evaluar_xy(X, y, umbral=0.5, quitar=(), semilla=42, mostrar=True):
    """Entrena un modelo nuevo y lo evalúa. Devuelve (precisión, recall, F1) de la clase 'atacante'.

    umbral : probabilidad mínima para marcar una IP como atacante.
    quitar : lista de características que NO se le darán al modelo.
    semilla: controla qué IPs caen en entrenamiento y cuáles en prueba.
    """
    cols = [c for c in X.columns if c not in quitar]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X[cols], y, test_size=0.25, stratify=y, random_state=semilla)
    m = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    m.fit(X_tr, y_tr)
    pred = (m.predict_proba(X_te)[:, 1] >= umbral).astype(int)
    p = precision_score(y_te, pred, zero_division=0)
    r = recall_score(y_te, pred, zero_division=0)
    f = f1_score(y_te, pred, zero_division=0)
    if mostrar:
        tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
        print(f"precisión={p:.3f}  recall={r:.3f}  F1={f:.3f}   "
              f"(bloqueos injustos={fp}, atacantes que se cuelan={fn})")
    return p, r, f

In [ ]:
evaluar_xy(X, y)   # resultado de referencia, sin tocar nada

## 6. 🧪 Experimentos

Para cada experimento, **antes de ejecutar** escribe en la tabla final tu **hipótesis** (¿qué crees que pasará y por qué?). Después ejecuta, anota el resultado y **explica** si coincidió con lo que esperabas.

Haz **al menos 5 de los 7**, incluyendo siempre E1 y E2.

### 🧪 E1 — El umbral de decisión
Cambia `UMBRAL` y vuelve a ejecutar. Prueba al menos 0.1, 0.3, 0.5, 0.8 y 0.95.

In [ ]:
UMBRAL = 0.5     # <-- cambia este valor
evaluar_xy(X, y, umbral=UMBRAL)

### 🧪 E2 — Quitar una característica
Cambia `QUITAR` y observa cuánto empeora (o mejora) el modelo. Prueba por ejemplo `["n_ok"]`, `["es_interna"]`, `["n_usuarios", "n_inv"]`, `["max_fallos_5min"]`.

In [ ]:
QUITAR = ["n_ok"]     # <-- cambia esta lista
evaluar_xy(X, y, quitar=QUITAR)

### 🧪 E3 — ¿Es mejor el modelo que una regla simple?
Una regla del tipo «si hay N o más fallos en 5 minutos, es un ataque». Cambia `N_FALLOS`.

In [ ]:
N_FALLOS = 5     # <-- cambia este valor
regla = (X_test["max_fallos_5min"] >= N_FALLOS).astype(int)
print("REGLA:")
print(f"  precisión={precision_score(y_test, regla, zero_division=0):.3f}  "
      f"recall={recall_score(y_test, regla, zero_division=0):.3f}  "
      f"F1={f1_score(y_test, regla, zero_division=0):.3f}")
print("MODELO:")
evaluar_xy(X, y)

### 🧪 E4 — ¿Cuánto depende el resultado de la suerte del reparto?
Se repite el experimento con 10 repartos distintos de entrenamiento/prueba. No hay nada que cambiar: observa la variación.

In [ ]:
resultados = [evaluar_xy(X, y, semilla=s, mostrar=False)[2] for s in range(10)]
print("F1 en 10 repartos:", [round(r, 3) for r in resultados])
print(f"media={pd.Series(resultados).mean():.3f}  mínimo={min(resultados):.3f}  máximo={max(resultados):.3f}")

### 🧪 E5 — ¿Funciona con datos nuevos?
Genera **otro** log con una semilla distinta en la terminal:

`python generar_logs.py --seed 999 --salida auth_b.log --etiquetas etiquetas_b.csv`

y comprueba cómo se comporta el modelo de la sección 4 (entrenado solo con tu log original) sobre estos datos que nunca ha visto.

In [ ]:
try:
    X_b = construir_features(cargar_log("auth_b.log", verbose=False))
    y_b = cargar_etiquetas("etiquetas_b.csv", X_b.index)
    pred_b = modelo.predict(X_b[X.columns])
    print(confusion_matrix(y_b, pred_b))
    print(classification_report(y_b, pred_b, digits=3))
except FileNotFoundError:
    print("Primero genera auth_b.log y etiquetas_b.csv con el comando de arriba.")

### 🧪 E6 — Muy pocos atacantes
En un servidor real, los ataques son una minoría pequeña. Genera un log con solo 12 atacantes:

`python generar_logs.py --seed 5 --atacantes 12 --salida auth_poco.log --etiquetas etiquetas_poco.csv`

In [ ]:
try:
    X_p = construir_features(cargar_log("auth_poco.log", verbose=False))
    y_p = cargar_etiquetas("etiquetas_poco.csv", X_p.index)
    print(f"Atacantes: {int(y_p.sum())} de {len(y_p)} IPs ({y_p.mean():.0%})")
    print(f"Atacantes que caen en el conjunto de prueba: {int(round(y_p.sum() * 0.25))}")
    evaluar_xy(X_p, y_p)
    # ¿Cuánto cambia el recall si cambia el reparto?
    recalls = [evaluar_xy(X_p, y_p, semilla=s, mostrar=False)[1] for s in range(10)]
    print("recall en 10 repartos:", [round(r, 2) for r in recalls])
except FileNotFoundError:
    print("Primero genera auth_poco.log y etiquetas_poco.csv con el comando de arriba.")

### 🧪 E7 — Detectar sin etiquetas
`IsolationForest` no mira `etiquetas.csv` para aprender: busca IPs «raras». Cambia `CONTAMINACION` (fracción de IPs que esperas que sean anómalas): prueba 0.05, 0.1, 0.25 y 0.4.

In [ ]:
CONTAMINACION = 0.25     # <-- cambia este valor
cols = ["n_eventos", "n_fallos", "n_usuarios", "n_inv",
        "fallos_por_min", "max_fallos_5min", "ratio_fallos"]
Z = StandardScaler().fit_transform(X[cols])
iso = IsolationForest(contamination=CONTAMINACION, random_state=42)
anomalo = (iso.fit_predict(Z) == -1).astype(int)

# Las etiquetas se usan SOLO aquí, para comprobar cómo lo ha hecho
print(confusion_matrix(y, anomalo))
print(classification_report(y, anomalo, digits=3))

## 7. Tabla de registro de experimentos

Copia esta tabla en tu informe y rellénala (una fila por cada prueba que hagas).

| Exp. | Qué cambié | Hipótesis (antes de ejecutar) | Resultado (precisión / recall / F1) | ¿Coincidió? Explicación |
|---|---|---|---|---|
| E1 | umbral = 0.1 | | | |
| E1 | umbral = 0.95 | | | |
| E2 | quité `n_ok` | | | |
| … | | | | |

❓ **P12.** Según tus experimentos, ¿qué umbral elegirías para un servidor con usuarios reales y por qué? ¿Y si el servidor guardara datos muy sensibles?

**Tu respuesta:**

❓ **P13.** ¿Qué experimento te ha enseñado algo que no esperabas? ¿Qué dice eso sobre fiarse de un único número de «acierto»?

**Tu respuesta:**